In [1]:
"""
Kaggle-Safe Ultra Optimized Inference Engine
Fully offline compatible. Production-grade.
"""

import os
import re
import gc
import random
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.cuda.amp import autocast
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

try:
    import sacrebleu
    _HAVE_SACREBLEU = True
except Exception:
    sacrebleu = None
    _HAVE_SACREBLEU = False

warnings.filterwarnings("ignore")

# ============================================================
# 🔒 Reproducibility
# ============================================================

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# ============================================================
# ⚙️ Config
# ============================================================

@dataclass
class Config:
    test_data_path: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    model_path: str = "/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x"
    output_dir: str = "/kaggle/working/"

    max_length: int = 512
    max_new_tokens: int = 512

    batch_size: int = 8
    num_workers: int = 2

    num_beams: int = 8
    length_penalty: float = 1.5
    early_stopping: bool = True

    # MBR decoding (single-idea upgrade: decoding-only reranking; same model)
    use_mbr: bool = True
    mbr_num_beam_cands: int = 4
    mbr_num_sample_cands: int = 2
    mbr_top_p: float = 0.9
    mbr_temperature: float = 0.7
    mbr_pool_cap: int = 16

    use_mixed_precision: bool = True
    use_adaptive_beams: bool = True
    use_bucket_batching: bool = True
    num_buckets: int = 4

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        self.use_mbr = bool(self.use_mbr)
        if not torch.cuda.is_available():
            self.use_mixed_precision = False

config = Config()

print(f"PyTorch: {torch.__version__}")
print(f"Device: {config.device}")

# ============================================================
# 📝 Logging
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

# ============================================================
# 🔄 Preprocessor
# ============================================================

class Preprocessor:
    def __init__(self):
        self.big_gap = re.compile(r"(\.{3,}|…+)")
        self.small_gap = re.compile(r"(xx+|\s+x\s+)")

    def process(self, texts: List[str]) -> List[str]:
        s = pd.Series(texts).fillna("").astype(str)
        s = s.str.replace(self.big_gap, "<big_gap>", regex=True)
        s = s.str.replace(self.small_gap, "<gap>", regex=True)
        return ("translate Akkadian to English: " + s).tolist()

# ============================================================
# 🧹 Postprocessor
# ============================================================

class Postprocessor:
    def __init__(self):
        self.whitespace = re.compile(r"\s+")
        self.repeated_words = re.compile(r"\b(\w+)(?:\s+\1\b)+")

    def process(self, texts: List[str]) -> List[str]:
        s = pd.Series(texts).fillna("").astype(str)
        s = s.str.replace(self.whitespace, " ", regex=True)
        s = s.str.replace(self.repeated_words, r"\1", regex=True)
        return s.str.strip().tolist()

# ============================================================
# 📦 Dataset
# ============================================================

class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor: Preprocessor):
        self.ids = df["id"].tolist()
        self.texts = preprocessor.process(df["transliteration"].tolist())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx]

# ============================================================
# 🪣 Bucket Sampler
# ============================================================

class BucketSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets):
        lengths = [len(t.split()) for _, t in dataset]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bucket_size = len(sorted_idx) // num_buckets

        self.batches = []
        for i in range(num_buckets):
            start = i * bucket_size
            end = None if i == num_buckets - 1 else (i + 1) * bucket_size
            bucket = sorted_idx[start:end]
            for j in range(0, len(bucket), batch_size):
                self.batches.append(bucket[j:j + batch_size])

    def __iter__(self):
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)

# ============================================================
# 🧠 Inference Engine
# ============================================================

class InferenceEngine:

    def __init__(self, config: Config):
        self.config = config
        self.pre = Preprocessor()
        self.post = Postprocessor()
        self._load_model()

    def _load_model(self):
        logger.info("Loading model (offline local path)...")

        model_path = Path(self.config.model_path)

        if not model_path.exists():
            raise ValueError(f"Model path not found: {model_path}")

        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_path,
            local_files_only=True,      # 🔑 prevents HF repo validation
            trust_remote_code=True
        ).to(self.config.device).eval()

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            local_files_only=True,
            trust_remote_code=True
        )

        total_params = sum(p.numel() for p in self.model.parameters())
        logger.info(f"Model loaded | Params: {total_params:,}")

    def _collate(self, batch):
        ids = [b[0] for b in batch]
        texts = [b[1] for b in batch]

        tokens = self.tokenizer(
            texts,
            max_length=self.config.max_length,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        return ids, tokens

    def _adaptive_beams(self, attention_mask):
        if not self.config.use_adaptive_beams:
            return self.config.num_beams

        lengths = attention_mask.sum(dim=1)
        if lengths.float().mean() < 80:
            return max(4, self.config.num_beams // 2)
        return self.config.num_beams

    def _dedup_preserve_order(self, texts: List[str]) -> List[str]:
        seen = set()
        unique = []
        for t in texts:
            if t not in seen:
                unique.append(t)
                seen.add(t)
        return unique

    def _char_ngram_f1(self, hyp: str, ref: str, n: int = 6) -> float:
        hyp = hyp or ""
        ref = ref or ""
        if not hyp or not ref:
            return 0.0

        n = int(max(1, min(n, len(hyp), len(ref))))
        hyp_ngrams = Counter(hyp[i : i + n] for i in range(len(hyp) - n + 1))
        ref_ngrams = Counter(ref[i : i + n] for i in range(len(ref) - n + 1))
        overlap = sum((hyp_ngrams & ref_ngrams).values())
        hyp_total = sum(hyp_ngrams.values())
        ref_total = sum(ref_ngrams.values())
        if hyp_total == 0 or ref_total == 0:
            return 0.0
        p = overlap / hyp_total
        r = overlap / ref_total
        if p + r == 0:
            return 0.0
        return 2 * p * r / (p + r)

    def _mbr_select(self, candidates: List[str]) -> str:
        if (not self.config.use_mbr) or len(candidates) <= 1:
            return candidates[0] if candidates else ""

        # Maximum Bayes Risk (MBR) via expected sentence-level BLEU over the candidate pool.
        use_sacrebleu = bool(_HAVE_SACREBLEU and sacrebleu is not None)
        scores = []
        for i, hyp in enumerate(candidates):
            s = 0.0
            for j, ref in enumerate(candidates):
                if i == j:
                    continue
                if use_sacrebleu:
                    s += sacrebleu.sentence_bleu(hyp, [ref]).score
                else:
                    s += 100.0 * self._char_ngram_f1(hyp, ref, n=6)
            scores.append(s)

        return candidates[int(np.argmax(scores))]

    def run(self, df: pd.DataFrame) -> pd.DataFrame:

        dataset = AkkadianDataset(df, self.pre)

        if self.config.use_bucket_batching:
            sampler = BucketSampler(dataset, self.config.batch_size, self.config.num_buckets)
            loader = DataLoader(
                dataset,
                batch_sampler=sampler,
                num_workers=self.config.num_workers,
                collate_fn=self._collate
            )
        else:
            loader = DataLoader(
                dataset,
                batch_size=self.config.batch_size,
                shuffle=False,
                num_workers=self.config.num_workers,
                collate_fn=self._collate
            )

        results = []

        with torch.inference_mode():
            for ids, tokens in tqdm(loader, desc="Translating"):
                input_ids = tokens.input_ids.to(self.config.device)
                attention_mask = tokens.attention_mask.to(self.config.device)

                beam_size = self._adaptive_beams(attention_mask)

                gen_kwargs = dict(
                    max_new_tokens=self.config.max_new_tokens,
                    num_beams=beam_size,
                    length_penalty=self.config.length_penalty,
                    early_stopping=self.config.early_stopping,
                    use_cache=True
                )

                if not self.config.use_mbr:
                    if self.config.use_mixed_precision:
                        with autocast():
                            outputs = self.model.generate(
                                input_ids=input_ids,
                                attention_mask=attention_mask,
                                **gen_kwargs
                            )
                    else:
                        outputs = self.model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            **gen_kwargs
                        )

                    decoded = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
                    cleaned = self.post.process(decoded)
                    results.extend(zip(ids, cleaned))
                    del outputs
                else:
                    k_beam = int(max(1, min(self.config.mbr_num_beam_cands, beam_size)))
                    k_sample = int(max(0, self.config.mbr_num_sample_cands))

                    beam_kwargs = dict(gen_kwargs)
                    beam_kwargs["num_return_sequences"] = k_beam

                    sample_kwargs = dict(
                        max_new_tokens=self.config.max_new_tokens,
                        do_sample=True,
                        top_p=self.config.mbr_top_p,
                        temperature=self.config.mbr_temperature,
                        num_return_sequences=k_sample,
                        use_cache=True,
                    )

                    if self.config.use_mixed_precision:
                        with autocast():
                            beam_out = self.model.generate(
                                input_ids=input_ids,
                                attention_mask=attention_mask,
                                **beam_kwargs
                            )
                            sample_out = (
                                self.model.generate(
                                    input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    **sample_kwargs
                                )
                                if k_sample > 0
                                else None
                            )
                    else:
                        beam_out = self.model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            **beam_kwargs
                        )
                        sample_out = (
                            self.model.generate(
                                input_ids=input_ids,
                                attention_mask=attention_mask,
                                **sample_kwargs
                            )
                            if k_sample > 0
                            else None
                        )

                    beam_dec = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)
                    beam_clean = self.post.process(beam_dec)
                    beam_groups = [
                        beam_clean[i * k_beam : (i + 1) * k_beam] for i in range(len(ids))
                    ]

                    if sample_out is not None:
                        sample_dec = self.tokenizer.batch_decode(sample_out, skip_special_tokens=True)
                        sample_clean = self.post.process(sample_dec)
                        sample_groups = [
                            sample_clean[i * k_sample : (i + 1) * k_sample] for i in range(len(ids))
                        ]
                    else:
                        sample_groups = [[] for _ in range(len(ids))]

                    for i, _id in enumerate(ids):
                        pool = beam_groups[i] + sample_groups[i]
                        pool = self._dedup_preserve_order(pool)[: self.config.mbr_pool_cap]
                        best = self._mbr_select(pool)
                        results.append((_id, best))

                    del beam_out, sample_out

                del input_ids, attention_mask

                torch.cuda.empty_cache()

        gc.collect()

        return pd.DataFrame(results, columns=["id", "translation"])

# ============================================================
# 🏁 Run
# ============================================================

logger.info("Loading test data...")
test_df = pd.read_csv(config.test_data_path)

engine = InferenceEngine(config)
submission = engine.run(test_df)

output_path = Path(config.output_dir) / "submission.csv"
submission.to_csv(output_path, index=False)

print("\nSubmission Preview:")
print(submission.head())
print(f"\nSaved to: {output_path}")

2026-02-13 05:57:29,652 | INFO | Loading test data...
2026-02-13 05:57:29,668 | INFO | Loading model (offline local path)...


PyTorch: 2.8.0+cu126
Device: cuda


2026-02-13 05:57:32.240777: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770962252.469087      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770962252.534438      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770962253.040458      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770962253.040493      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770962253.040496      25 computation_placer.cc:177] computation placer alr

Translating:   0%|          | 0/4 [00:00<?, ?it/s]


Submission Preview:
   id                                        translation
0   0  Thus says the Kanesh colony: Speak to our mess...
1   3  I sent our tablet to every single colony and t...
2   1  As for the tablet of the City, you wrote to me...
3   2  As soon as you hear our letter, there he has g...

Saved to: /kaggle/working/submission.csv
